# Lenguajes y Autómatas II (SCD-1016)
## Sesión 27 · Indexación de Arreglos en Código Intermedio

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se convierte un acceso A[i] en una dirección de memoria concreta?
>
> **Objetivo de la sesión:** Calcular el desplazamiento de un índice, sumarlo a la base y emitir las instrucciones TAC de lectura indexada para arreglos de una y dos dimensiones.

**Entregable:** tus funciones emitir_acceso() y emitir_acceso_2d() en artesano_sesion27_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion27.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Para un arreglo 1D emites la multiplicación del índice por el ancho y la lectura base[_tK]. Para uno 2D primero aplanas i * filas + j, luego multiplicas por el ancho y lees. Los temporales se numeran desde *temporales.

Al final de la sesión tendrás: **Dos funciones en C que emiten el cálculo de offset y la lectura indexada para arreglos 1D y 2D.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 27](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion27.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion27.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion27.h"

int emitir_acceso(char lineas[][64], int max, const char *base, const char *indice, int ancho, int *temporales);
int emitir_acceso_2d(char lineas[][64], int max, const char *base, const char *i, const char *j, int filas, int ancho, int *temporales);
```

- Multiplica siempre el índice por el ancho del elemento antes de usarlo como offset.
- En 2D respeta el orden i * filas + j; intercambiar dimensiones lee otra celda.
- Cada temporal nuevo incrementa *temporales; no repitas nombres.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion27_c.c` junto con `artesano_sesion27_c.c`
> (o `ia_sesion27_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `1D: i * 4` | `_t0 = i * 4` |
| 2 | `1D: A[_t0]` | `_t1 = A[_t0]` |
| 3 | `2D: i * 3` | `_t0 = i * 3` |
| 4 | `2D: offset por ancho` | `_t2 = _t1 * 4` |
| 5 | `2D: M[_t2]` | `_t3 = M[_t2]` |
| 6 | `base NULL (sin acceso)` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 27 - Desplazamientos para arreglos 1D y 2D ===
Caso                               esperado:              obtenido:              resultado
[ 1] 1D: i * 4                          esperado: _t0 = i * 4  obtenido: _t0 = i * 4  OK
[ 2] 1D: A[_t0]                         esperado: _t1 = A[_t0] obtenido: _t1 = A[_t0] OK
[ 3] 2D: i * 3                          esperado: _t0 = i * 3  obtenido: _t0 = i * 3  OK
[ 4] 2D: offset por ancho
     esperado: _t2 = _t1 * 4
     obtenido: _t2 = _t1 * 4
     -> OK
[ 5] 2D: M[_t2]                         esperado: _t3 = M[_t2] obtenido: _t3 = M[_t2] OK
[ 6] base NULL (sin acceso)             esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion27.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El offset siempre incluye la multiplicación por el ancho del elemento.
- [ ] En 2D el índice lineal es i * filas + j, en ese orden.
- [ ] Cada acceso consume temporales nuevos y una base NULL no emite instrucciones.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion27_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion27.h
/* api_sesion27.h - contrato de la Sesion 27
 * Desplazamientos para arreglos 1D y 2D
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION27_H
#define API_SESION27_H

int emitir_acceso(char lineas[][64], int max, const char *base, const char *indice, int ancho, int *temporales);
int emitir_acceso_2d(char lineas[][64], int max, const char *base, const char *i, const char *j, int filas, int ancho, int *temporales);

#endif /* API_SESION27_H */


In [ ]:
%%writefile artesano_sesion27_c.c
/* artesano_sesion27_c.c - Fase 1: El Artesano (sin IA)
 * Desplazamientos para arreglos 1D y 2D.
 *
 * Implementa lo que declara api_sesion27.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion27.h"
#include <stdio.h>
#include <string.h>

int emitir_acceso(char lineas[][64], int max, const char *base, const char *indice, int ancho, int *temporales) {
          (void)lineas; (void)max; (void)base; (void)indice; (void)ancho; (void)temporales; /* TODO: offset y lectura */
          return 0;
      }
      
      int emitir_acceso_2d(char lineas[][64], int max, const char *base, const char *i, const char *j, int filas, int ancho, int *temporales) {
          (void)lineas; (void)max; (void)base; (void)i; (void)j; (void)filas; (void)ancho; (void)temporales; /* TODO: aplanar y leer */
          return 0;
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion27_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion27_c.c
/* ia_sesion27_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion27.h. No puede traer main().
 */
#include "api_sesion27.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion27_c.c
/* harness_sesion27_c.c - auditoria automatica de la Sesion 27
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion27_c.c artesano_sesion27_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion27.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 27 - Desplazamientos para arreglos 1D y 2D ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char l[8][64] = {{0}}; int t = 0; emitir_acceso(l, 8, "A", "i", 4, &t); verificar_str("1D: i * 4", "_t0 = i * 4", l[0]); }
    { char l[8][64] = {{0}}; int t = 0; emitir_acceso(l, 8, "A", "i", 4, &t); verificar_str("1D: A[_t0]", "_t1 = A[_t0]", l[1]); }
    { char l[16][64] = {{0}}; int t = 0; emitir_acceso_2d(l, 16, "M", "i", "j", 3, 4, &t); verificar_str("2D: i * 3", "_t0 = i * 3", l[0]); }
    { char l[16][64] = {{0}}; int t = 0; emitir_acceso_2d(l, 16, "M", "i", "j", 3, 4, &t); verificar_str_largo("2D: offset por ancho", "_t2 = _t1 * 4", l[2]); }
    { char l[16][64] = {{0}}; int t = 0; emitir_acceso_2d(l, 16, "M", "i", "j", 3, 4, &t); verificar_str("2D: M[_t2]", "_t3 = M[_t2]", l[3]); }
    { char l[8][64] = {{0}}; int t = 0; int k = emitir_acceso(l, 8, NULL, "i", 4, &t); verificar_int("base NULL (sin acceso)", (long)(0), (long)(k)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion27.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion27_c.c artesano_sesion27_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion27_c.c ia_sesion27_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion27.md` a `auditorias/auditoria_sesion27.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
